In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "SUNGA"
STUDENT_ID = "TUPM-26-1215"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

assert measurement_path.is_file()
assert label_path.is_file()

raw_measurements = np.genfromtxt(
    measurement_path,
    dtype=float
)

labels = np.loadtxt(
    label_path,
    usecols=0,
    dtype=int
)

assert len(labels) == len(raw_measurements)

df = pd.DataFrame(
    raw_measurements,
    columns=[
        f"Feature_{i:03d}"
        for i in range(raw_measurements.shape[1])
    ]
)

df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

observed = np.sum(~np.isnan(raw_measurements), axis=0)

all_missing = observed == 0

minimum = np.where(
    np.isnan(raw_measurements),
    np.inf,
    raw_measurements
).min(axis=0)

maximum = np.where(
    np.isnan(raw_measurements),
    -np.inf,
    raw_measurements
).max(axis=0)

constant = (
    (observed > 0) &
    (minimum == maximum)
)

usable_indices = np.flatnonzero(
    ~(all_missing | constant)
)

retained_features = [
    f"Feature_{i:03d}" for i in usable_indices
]

print("Student:", LAST_NAME, STUDENT_ID)
print("Random Seed:", RANDOM_SEED)
print("Dataset Shape:", raw_measurements.shape)
print("Usable Features:", len(retained_features))
print("Pass Records:", np.sum(labels == -1))
print("Fail Records:", np.sum(labels == 1))
print("Output Folder:", OUTPUT_DIR.resolve())


Student: SUNGA TUPM-26-1215
Random Seed: 261215
Dataset Shape: (1567, 590)
Usable Features: 474
Pass Records: 1463
Fail Records: 104
Output Folder: C:\Users\123\Lab7_NumPy_Pandas_Sunga\Lab7_NumPy_Pandas_Additional_Exercises\output


In [2]:

rng = np.random.default_rng(RANDOM_SEED)

selected_count = 5 + (SEED_DIGIT % 3)

selected_positions = rng.choice(
    usable_indices,
    size=selected_count,
    replace=False
)

selected_features = [
    f"Feature_{i:03d}" for i in selected_positions
]

quality_data = raw_measurements[:, selected_positions]

quality_median = np.nanmedian(quality_data, axis=0)

quality_mean = np.nanmean(quality_data, axis=0)

quality_sd = np.nanstd(
    quality_data,
    axis=0,
    ddof=1
)

missing_percentage = (
    np.isnan(quality_data).mean(axis=0) * 100
)

mad = np.nanmedian(
    np.abs(quality_data - quality_median),
    axis=0
)

safe_mad = np.where(mad > 0, mad, 1.0)

robust_z = (
    0.6745 * (quality_data - quality_median) / safe_mad
)

robust_z = np.where(
    np.isnan(quality_data),
    np.nan,
    robust_z
)

flagged = np.abs(robust_z) > 3.5

flagged_counts = np.sum(flagged, axis=0)

flagged_percentage = (
    flagged_counts / observed[selected_positions] * 100
)

measurement_quality = pd.DataFrame({
    "Feature": selected_features,
    "Mean": quality_mean,
    "Median": quality_median,
    "Sample_SD": quality_sd,
    "Missing_Percentage": missing_percentage,
    "MAD": mad,
    "Flagged_Observations": flagged_counts,
    "Flagged_Percentage": flagged_percentage
})

measurement_quality = measurement_quality.sort_values(
    "Flagged_Percentage",
    ascending=False
).reset_index(drop=True)

quality_path = OUTPUT_DIR / "measurement_quality.csv"

measurement_quality.to_csv(
    quality_path,
    index=False
)

highest_missing = np.argmax(missing_percentage)
highest_flagged = np.argmax(flagged_percentage)

print("Selected Features:", selected_features)
print(
    "Highest Missing Percentage:",
    f"{missing_percentage[highest_missing]:.2f}%"
)
print("Feature with Highest Flagged Percentage:",
      selected_features[highest_flagged])
print("Total Flagged Observations:", flagged_counts.sum())
print("Vectorized Results Verified:",
      measurement_quality.shape[0] == selected_count)
print("\nMeasurement Quality Summary:")
print(measurement_quality.round(4))
print("\nExported:", quality_path.name)


Selected Features: ['Feature_098', 'Feature_347', 'Feature_032', 'Feature_066', 'Feature_018', 'Feature_286', 'Feature_294']
Highest Missing Percentage: 0.38%
Feature with Highest Flagged Percentage: Feature_294
Total Flagged Observations: 216
Vectorized Results Verified: True

Measurement Quality Summary:
       Feature      Mean    Median  Sample_SD  Missing_Percentage       MAD  \
0  Feature_294  401.8148  278.6719   477.0501              0.1276  109.9702   
1  Feature_098   -0.0181    0.0000     0.4271              0.3829    0.1975   
2  Feature_032   85.3375   85.1354     2.0265              0.0638    0.6192   
3  Feature_286    4.2600    3.3600     9.7638              0.1914    0.9200   
4  Feature_018  190.0474  189.6642     2.7810              0.1914    1.8763   
5  Feature_066  706.6685  706.4536    11.6231              0.3829    7.8761   
6  Feature_347    0.0089    0.0000     0.3522              0.3829    0.0000   

   Flagged_Observations  Flagged_Percentage  
0            

In [3]:

rng = np.random.default_rng(RANDOM_SEED)

pass_positions = np.flatnonzero(labels == -1)
fail_positions = np.flatnonzero(labels == 1)

balanced_per_group = min(
    20 + ID_SUM,
    len(pass_positions),
    len(fail_positions)
)

selected_features_2 = rng.choice(
    retained_features,
    size=5,
    replace=False
).tolist()

pass_sample = rng.choice(
    pass_positions,
    size=balanced_per_group,
    replace=False
)

fail_sample = rng.choice(
    fail_positions,
    size=balanced_per_group,
    replace=False
)

balanced_positions = np.concatenate([
    pass_sample,
    fail_sample
])

rng.shuffle(balanced_positions)

balanced_sample = df.iloc[balanced_positions].copy()

balanced_sample.insert(
    0,
    "Example_ID",
    balanced_positions
)

# Use full-dataset medians for imputation
full_medians = df[retained_features].median()

balanced_sample[selected_features_2] = (
    balanced_sample[selected_features_2]
    .fillna(full_medians[selected_features_2])
)

comparison_rows = []

for feature in selected_features_2:

    pass_values = balanced_sample.loc[
        balanced_sample["Outcome"] == "Pass",
        feature
    ]

    fail_values = balanced_sample.loc[
        balanced_sample["Outcome"] == "Fail",
        feature
    ]

    pass_mean = pass_values.mean()
    fail_mean = fail_values.mean()

    pass_median = pass_values.median()
    fail_median = fail_values.median()

    pass_sd = pass_values.std(ddof=1)
    fail_sd = fail_values.std(ddof=1)

    pooled_sd = np.sqrt(
        (
            (len(pass_values) - 1) * pass_sd**2
            +
            (len(fail_values) - 1) * fail_sd**2
        )
        /
        (len(pass_values) + len(fail_values) - 2)
    )

    absolute_difference = abs(fail_mean - pass_mean)

    standardized_difference = (
        absolute_difference / pooled_sd
        if pooled_sd > 0 else 0.0
    )

    comparison_rows.append({
        "Feature": feature,
        "Pass_Count": len(pass_values),
        "Fail_Count": len(fail_values),
        "Pass_Mean": pass_mean,
        "Fail_Mean": fail_mean,
        "Pass_Median": pass_median,
        "Fail_Median": fail_median,
        "Pass_SD": pass_sd,
        "Fail_SD": fail_sd,
        "Absolute_Mean_Difference": absolute_difference,
        "Pooled_SD": pooled_sd,
        "Standardized_Mean_Difference": standardized_difference
    })

feature_comparison = pd.DataFrame(comparison_rows)

feature_comparison = feature_comparison.sort_values(
    "Standardized_Mean_Difference",
    ascending=False
).reset_index(drop=True)

balanced_path = OUTPUT_DIR / "balanced_sample.csv"
comparison_path = OUTPUT_DIR / "feature_comparison.csv"

balanced_sample.to_csv(
    balanced_path,
    index=False
)

feature_comparison.to_csv(
    comparison_path,
    index=False
)

print("Balanced Sample Size:", len(balanced_sample))
print("Records per Outcome:", balanced_per_group)
print("Selected Features:", selected_features_2)

print(
    "Highest-Ranked Feature:",
    feature_comparison.iloc[0]["Feature"]
)

print(
    "Highest Standardized Mean Difference:",
    f"{feature_comparison.iloc[0]['Standardized_Mean_Difference']:.4f}"
)

print(
    "NumPy-Pandas Statistics Match:",
    np.isclose(
        balanced_sample[selected_features_2].to_numpy().mean(),
        balanced_sample[selected_features_2].to_numpy().mean()
    )
)

print("\nFeature Comparison:")
print(feature_comparison.round(4))

print("\nExported:", balanced_path.name)
print("Exported:", comparison_path.name)


Balanced Sample Size: 74
Records per Outcome: 37
Selected Features: ['Feature_295', 'Feature_032', 'Feature_066', 'Feature_287', 'Feature_018']
Highest-Ranked Feature: Feature_287
Highest Standardized Mean Difference: 0.4659
NumPy-Pandas Statistics Match: True

Feature Comparison:
       Feature  Pass_Count  Fail_Count  Pass_Mean  Fail_Mean  Pass_Median  \
0  Feature_287          37          37     0.1619     0.2655       0.1405   
1  Feature_066          37          37   703.5903   708.3368     703.4975   
2  Feature_295          37          37   245.0800   403.0556     195.3885   
3  Feature_018          37          37   190.3033   189.8252     190.6226   
4  Feature_032          37          37    85.3879    85.5889      85.2505   

   Fail_Median   Pass_SD   Fail_SD  Absolute_Mean_Difference  Pooled_SD  \
0       0.1685    0.0673    0.3069                    0.1035     0.2222   
1     706.6697   11.9380   11.5324                    4.7466    11.7369   
2     233.6370  292.1520  487.

In [4]:
numpy_means = balanced_sample[selected_features_2].to_numpy().mean(axis=0)

pandas_means = balanced_sample[selected_features_2].mean().to_numpy()

print(
    "NumPy-Pandas Statistics Match:",
    np.allclose(numpy_means, pandas_means)
)

NumPy-Pandas Statistics Match: True


In [5]:

rng = np.random.default_rng(RANDOM_SEED)

# Select five features with at least 10 observed values
# in both Pass and Fail groups.

eligible_features = []

for feature in retained_features:

    pass_count_observed = df.loc[
        df["Outcome"] == "Pass", feature
    ].notna().sum()

    fail_count_observed = df.loc[
        df["Outcome"] == "Fail", feature
    ].notna().sum()

    if pass_count_observed >= 10 and fail_count_observed >= 10:
        eligible_features.append(feature)

assert len(eligible_features) >= 5

selected_features_3 = rng.choice(
    eligible_features,
    size=5,
    replace=False
).tolist()

BOOTSTRAP_REPETITIONS = 200 + 10 * SEED_DIGIT

# Preparation strategy 1: available-case analysis
# Preparation strategy 2: full-dataset median imputation

full_data_medians = df[retained_features].median()

imputed_data = df[selected_features_3].fillna(
    full_data_medians[selected_features_3]
)

def calculate_difference(pass_values, fail_values):
    return fail_values.mean() - pass_values.mean()

def stratified_bootstrap(pass_values, fail_values, seed):

    local_rng = np.random.default_rng(seed)

    differences = []

    pass_array = np.asarray(pass_values)
    fail_array = np.asarray(fail_values)

    for _ in range(BOOTSTRAP_REPETITIONS):

        pass_boot = local_rng.choice(
            pass_array,
            size=len(pass_array),
            replace=True
        )

        fail_boot = local_rng.choice(
            fail_array,
            size=len(fail_array),
            replace=True
        )

        differences.append(
            fail_boot.mean() - pass_boot.mean()
        )

    differences = np.asarray(differences)

    lower = np.percentile(differences, 2.5)
    upper = np.percentile(differences, 97.5)

    return differences.mean(), lower, upper

results = []

for feature_index, feature in enumerate(selected_features_3):

    for strategy in ["Available Case", "Median Imputation"]:

        if strategy == "Available Case":

            pass_values = df.loc[
                df["Outcome"] == "Pass", feature
            ].dropna().to_numpy()

            fail_values = df.loc[
                df["Outcome"] == "Fail", feature
            ].dropna().to_numpy()

        else:

            pass_values = imputed_data.loc[
                df["Outcome"] == "Pass", feature
            ].to_numpy()

            fail_values = imputed_data.loc[
                df["Outcome"] == "Fail", feature
            ].to_numpy()

        observed_difference = calculate_difference(
            pass_values,
            fail_values
        )

        bootstrap_mean, lower, upper = stratified_bootstrap(
            pass_values,
            fail_values,
            RANDOM_SEED + feature_index
        )

        results.append({
            "Feature": feature,
            "Strategy": strategy,
            "Pass_Mean": pass_values.mean(),
            "Fail_Mean": fail_values.mean(),
            "Mean_Difference": observed_difference,
            "Bootstrap_Mean_Difference": bootstrap_mean,
            "CI_95_Lower": lower,
            "CI_95_Upper": upper
        })

robustness_comparison = pd.DataFrame(results)

# Rank features separately for each preparation strategy
robustness_comparison["Absolute_Mean_Difference"] = (
    robustness_comparison["Mean_Difference"].abs()
)

robustness_comparison["Feature_Rank"] = (
    robustness_comparison.groupby("Strategy")[
        "Absolute_Mean_Difference"
    ].rank(ascending=False, method="min")
)

# Compare the two strategies
available = (
    robustness_comparison[
        robustness_comparison["Strategy"] == "Available Case"
    ].set_index("Feature")
)

imputed = (
    robustness_comparison[
        robustness_comparison["Strategy"] == "Median Imputation"
    ].set_index("Feature")
)

merged = available.join(
    imputed,
    lsuffix="_Available",
    rsuffix="_Imputed"
)

sign_changes = (
    np.sign(merged["Mean_Difference_Available"])
    != np.sign(merged["Mean_Difference_Imputed"])
)

rank_changes = (
    merged["Feature_Rank_Available"]
    != merged["Feature_Rank_Imputed"]
)

mean_changes = (
    merged["Mean_Difference_Available"]
    - merged["Mean_Difference_Imputed"]
).abs()

largest_change = mean_changes.max()

# Reproducibility check
repeat_rng = np.random.default_rng(RANDOM_SEED)

repeat_features = repeat_rng.choice(
    eligible_features,
    size=5,
    replace=False
).tolist()

reproducibility_verified = (
    repeat_features == selected_features_3
)

robustness_path = OUTPUT_DIR / "robustness_comparison.csv"
sensitivity_path = OUTPUT_DIR / "sensitivity_report.txt"

robustness_comparison.to_csv(
    robustness_path,
    index=False
)

sensitivity_report = (
    "STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS\n"
    f"Student: {LAST_NAME}\n"
    f"Student ID: {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Selected Features: {', '.join(selected_features_3)}\n"
    f"Bootstrap Repetitions: {BOOTSTRAP_REPETITIONS}\n"
    f"Largest Change in Mean Difference: {largest_change:.6f}\n"
    f"Features with Changed Difference Sign: {sign_changes.sum()}\n"
    f"Features with Changed Ranking: {rank_changes.sum()}\n"
    f"Reproducibility Verified: {reproducibility_verified}\n"
    "\nInterpretation:\n"
    "- Available-case analysis uses only observed measurements.\n"
    "- Median imputation replaces missing values using full-data medians.\n"
    "- Bootstrap confidence intervals describe sampling uncertainty.\n"
    "- Differences between strategies indicate sensitivity to preparation.\n"
)

sensitivity_path.write_text(
    sensitivity_report,
    encoding="utf-8"
)

print("Selected Features:", selected_features_3)
print("Bootstrap Repetitions:", BOOTSTRAP_REPETITIONS)
print("Largest Change in Mean Difference:", f"{largest_change:.6f}")
print("Features with Changed Difference Sign:", sign_changes.sum())
print("Features with Changed Ranking:", rank_changes.sum())
print("Reproducibility Verified:", reproducibility_verified)

print("\nRobustness Comparison:")
print(robustness_comparison.round(4))

print("\nExported:", robustness_path.name)
print("Exported:", sensitivity_path.name)


Selected Features: ['Feature_297', 'Feature_032', 'Feature_066', 'Feature_287', 'Feature_018']
Bootstrap Repetitions: 250
Largest Change in Mean Difference: 13.139245
Features with Changed Difference Sign: 0
Features with Changed Ranking: 0
Reproducibility Verified: True

Robustness Comparison:
       Feature           Strategy  Pass_Mean  Fail_Mean  Mean_Difference  \
0  Feature_297     Available Case  2346.1919  2295.0653         -51.1265   
1  Feature_297  Median Imputation  2345.1487  2280.8830         -64.2658   
2  Feature_032     Available Case    85.3093    85.7335           0.4242   
3  Feature_032  Median Imputation    85.3092    85.7335           0.4243   
4  Feature_066     Available Case   706.7897   704.9710          -1.8187   
5  Feature_066  Median Imputation   706.7883   704.9710          -1.8173   
6  Feature_287     Available Case     0.3770     0.2347          -0.1423   
7  Feature_287  Median Imputation     0.3765     0.2347          -0.1418   
8  Feature_018     A

C:\Users\123\AppData\Local\Temp\ipykernel_1772\2329995681.py:151: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  merged = available.join(
